In [ ]:
import numpy as np
import pandas as pd
from modelens import RegressionAnalyzer, regression_models

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_csv("dataset.csv")
df = raw_data.copy()

#### 1 - EDA + ETL

In [ ]:
analyzer = RegressionAnalyzer(df, target="mpg")

In [ ]:
analyzer.info()

In [ ]:
# Check Invalid Data
df["horsepower"].unique()

In [ ]:
# ? is Invalida Data in horsepower Column
df["horsepower"] = df["horsepower"].replace({"?": np.nan}).astype(float)
df = df.dropna(subset=["horsepower"])
df["horsepower"].unique()

In [ ]:
# One Hot Encoding
df = pd.get_dummies(
    df,
    columns=["origin"],
    prefix="origin",
    drop_first=True,
    dtype=int,
)

In [ ]:
# Remove UnUsed Columns
if "car name" in df.columns:
    df = df.drop(columns=["car name"])

In [ ]:
analyzer.reinit(df=df, target="mpg")

In [ ]:
analyzer.info()

In [ ]:
df.head()

#### 2 - Comparing Models

In [ ]:
target = "mpg"
X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop(target)

In [ ]:
models = regression_models()

# analyzer.compare_models(
#     models=models,
#     features=features,
#     export_html=True,
#     file_name="01-base-model-comparing",
# )

#### Model Selection for Tuning
***1. CatBoost ***: بهترین ترین و تست

***2. Polynomial Regression D2 *** بهترین سرعت فیت و پردیکت

***3. LightGBM *** بهترین سرعت فیت

***4. Gradient Boosting *** بهترین سرعت پردیکت

In [ ]:
selected_models = regression_models(
    smodels=[
        # "CatBoost",
        "Polynomial Regression",
        "LightGBM",
        "Gradient Boosting",
    ]
)


analyzer.compare_models(
    models=selected_models,
    features=features,
    export_html=True,
    file_name="02-selected-model",
)

In [ ]:
param_grids = {
    # "CatBoost": {
    #     "iterations": [300, 500, 1000],
    #     # "depth": [4, 6, 8],
    #     # "learning_rate": [0.03, 0.05, 0.1],
    #     # "l2_leaf_reg": [1, 3, 5],
    # },
    "Polynomial Regression": {
        "polynomial__degree": [2, 3, 4],
    },
    "LightGBM": {
        "n_estimators": [100, 300, 500],
        # "learning_rate": [0.03, 0.1],
        # "num_leaves": [15, 31, 63],
        # "max_depth": [-1, 10],
        # "min_child_samples": [10, 20],
        # "colsample_bytree": [0.8, 1.0],
    },
    "Gradient Boosting": {
        "n_estimators": [100, 300, 500],
        # "learning_rate": [0.03, 0.05, 0.1],
        # "max_depth": [2, 3, 5],
        # "min_samples_split": [2, 5],
        # "min_samples_leaf": [1, 2, 4],
    },
}

# for name, model in selected_models.items():

#     analyzer.tune_model(
#         model=model,
#         features=features,
#         param_grid=param_grids[name],
#         export_html=True,
#         file_name=f"{name.lower().replace(' ', '-')}",
#     )

#### Decrease Feature

In [ ]:
_, suspicious_features = analyzer.correlation(features=features)

In [ ]:
analyzer.vif(features=features)

In [ ]:
from catboost import CatBoostRegressor

selected_models["CatBoost"] = CatBoostRegressor(
    depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05
)

In [ ]:
# for name, model in selected_models.items():
#     analyzer.evaluate_single_feature_removal(
#         model=model,
#         features=features,
#         export_html=True,
#         file_name=f"{name.lower().replace(' ', '-')}",
#     )

In [ ]:
# for name, model in selected_models.items():
#     analyzer.evaluate_feature_removal_combinations(
#         model=model,
#         features=features,
#         candidates=suspicious_features,
#         export_html=True,
#         file_name=f"{name.lower().replace(' ', '-')}",
#     )

In [ ]:
selected_model = models["CatBoost"]

In [ ]:
selected_features = [
    "model year",
    "weight",
]

analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "All Features": features,
        "Model Year + Weight": selected_features,
    },
    export_html=True,
    file_name="catboost_feature_comparison",
)

In [ ]:
analyzer.residual_analysis(model=selected_model,export_html=True)

In [ ]:
analyzer.learning_curve(model=selected_model)